In [2]:
import os
import pickle
import ijson
import jsonlines
import pandas as pd
import numpy as np
import json
import copy
from tqdm import tqdm
from collections import defaultdict

In [3]:
try:
    META_FILE = "../../output/yelp/item2attributes.json"   # 相对 notebook 的新路径
    data = json.load(open(META_FILE, "r", encoding="utf-8"))
    print(f"Loaded metadata for {len(data)} items from {META_FILE}")
except FileNotFoundError:
    print(f"Error: {META_FILE} not found. Did grocery_data_process.py run successfully?")
    exit()
except json.JSONDecodeError:
    print(f"Error: {META_FILE} is not a valid JSON file.")
    exit()

Loaded metadata for 112394 items from ../../output/yelp/item2attributes.json


In [4]:
example_dict = {}
for item_dict in tqdm(data.values()):
    example_dict.update(item_dict)
example_dict.keys()

100%|██████████| 112394/112394 [00:00<00:00, 575016.47it/s]


dict_keys(['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours'])

保留 LLMEmb 的提示思路：
* 统一开头："The point of interest has the following attributes: 
"
* 去除经纬度 (longitude / latitude) 字段
* 文本字段：直接 `key: value` 追加
* 列表字段：将列表元素以逗号连接为字符串后追加
* 最终保存 `business_id -> prompt str` JSON

In [5]:
instruction = "The point of interest has the following attributes: \n"

In [ ]:
item_data = {}
for item_dict in tqdm(data.values()):
    item_prompt = copy.deepcopy(instruction)
    item_id = None
    for key, value in item_dict.items():
        if key in ["longitude", "latitude",]:   # drop longitude and latitude
            continue
        elif key in ["business_id"]:  # get the item id
            item_id = value
        elif key in ["categories", "neighborhoods"]:    # list type attributes
            attri_str = ""
            if isinstance(value, list):                 # 仅当 value 是 list 再遍历
                for meta_str in value:
                    attri_str += meta_str + ", "
                if len(value) == 0:
                    attri_str = "none, "
                attri_prompt = key + " is " + attri_str[:-2] + "; "    # [:-2] is to remove the last ", "
                item_prompt += attri_prompt
        else:   # str type attributes 
            attri_prompt = key + " is " + str(value).replace("\n", ", ") + "; "
            item_prompt += attri_prompt
    if item_id:
        item_data[item_id] = item_prompt[:-2]
    else:
        raise ValueError("No item id")

 10%|█         | 11306/112394 [00:00<00:03, 28752.53it/s]


TypeError: object of type 'NoneType' has no len()

In [ ]:
json.dump(item_data, open("../../output/yelp/item_str.json", "w"))